# 🏥 Clinical AI — Oral Lesion Classification

Questo notebook riproduce l'intera pipeline end-to-end su Google Colab.

**Steps:**
1. Clone del repository
2. Installazione dipendenze
3. Verifica GPU
4. Esecuzione test suite (`pytest`)
5. Debug run (dati sintetici, nessun dataset reale richiesto)
6. *(Opzionale)* Carica il tuo dataset e lancia la pipeline completa

> **Tempo stimato:** ~5-8 minuti su Colab T4 (incluso install dipendenze)

## 1 · Clone del repository

In [ ]:
import os

REPO_URL = "https://github.com/fabiooraziomirto/ML-Bitto.git"
REPO_DIR = "/content/ML-Bitto"

if os.path.isdir(REPO_DIR):
    print("Repo già presente — aggiornamento...")
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"\nWorking directory: {os.getcwd()}")

## 2 · Installazione dipendenze

Colab include già PyTorch con CUDA. Installiamo le restanti dipendenze del progetto senza sovrascrivere torch/torchvision già presenti.

In [ ]:
# Installa le dipendenze del progetto mantenendo torch/torchvision di Colab
!pip install -q \
    numpy==2.4.4 \
    pandas==3.0.2 \
    scikit-learn==1.8.0 \
    Pillow==12.2.0 \
    opencv-python==4.13.0.92 \
    PyYAML==6.0.3 \
    python-dotenv==1.2.2 \
    tqdm==4.67.3 \
    matplotlib==3.10.1 \
    pytest==9.0.2

print("\n✓ Dipendenze installate")

## 3 · Verifica GPU e versioni

In [ ]:
import torch
import torchvision
import numpy as np
import pandas as pd
import sklearn

print("=" * 50)
print("  Ambiente di esecuzione")
print("=" * 50)
print(f"  PyTorch      : {torch.__version__}")
print(f"  Torchvision  : {torchvision.__version__}")
print(f"  NumPy        : {np.__version__}")
print(f"  Pandas       : {pd.__version__}")
print(f"  scikit-learn : {sklearn.__version__}")
print()

device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"  GPU          : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("  ⚠️  GPU non disponibile — esecuzione su CPU (più lenta)")
    print("     Vai su Runtime > Cambia tipo di runtime > GPU T4")

print(f"\n  Dispositivo attivo: {device}")
print("=" * 50)

## 4 · Test suite

Esegue `pytest tests.py` per verificare che tutti i moduli siano correttamente importabili e che la logica di splitting/loading sia intatta.

In [ ]:
!python -m pytest tests.py -v --tb=short 2>&1

## 5 · Debug run — pipeline completa con dati sintetici

Esegue la pipeline end-to-end (load → split → train → evaluate) su dati sintetici generati in memoria.  
**Non richiede nessun dataset reale.**

- 60 campioni sintetici, 3 epoche
- Early stopping attivo (metrica: recall)
- Produce metriche finali + confusion matrix + ROC curve in `evaluation_results/`

In [ ]:
!python main.py --debug --monitor recall 2>&1

### Visualizza risultati della debug run

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

results_dir = Path("evaluation_results")

# --- Metriche JSON ---
json_path = results_dir / "metrics_report.json"
if json_path.exists():
    with open(json_path) as f:
        report = json.load(f)
    print("\n📊 Metriche finali — AI vs Biopsy")
    print("-" * 40)
    for k, v in report.get("AI vs Biopsy", {}).items():
        print(f"  {k:<18}: {v}")
else:
    print("[INFO] metrics_report.json non trovato — la debug run potrebbe non aver completato la valutazione.")

# --- Plot immagini generate ---
plots = {
    "ROC Curve": results_dir / "roc_curve.png",
    "Confusion Matrix": results_dir / "confusion_matrices.png",
    "Calibration Curve": results_dir / "calibration_curve.png",
}

existing = {name: path for name, path in plots.items() if path.exists()}

if existing:
    fig, axes = plt.subplots(1, len(existing), figsize=(7 * len(existing), 6))
    if len(existing) == 1:
        axes = [axes]
    for ax, (name, path) in zip(axes, existing.items()):
        ax.imshow(mpimg.imread(path))
        ax.set_title(name, fontsize=13, fontweight="bold")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("\n[INFO] Nessun plot trovato (normale se la debug run usa troppo pochi campioni per ROC).")

---

## 6 · (Opzionale) Pipeline con dataset reale

Carica il tuo dataset su Colab e modifica i path qui sotto.  
**Struttura attesa** — dataset pubblico:
```
data/public_dataset_1/
├── images/
│   ├── img_001.jpg
│   └── ...
└── labels.csv        # colonne: image_name, label
```

**Struttura attesa** — dataset clinico:
```
data/clinical_dataset/
├── images/
└── metadata.csv      # colonne: image_name, biopsy_diagnosis [, patient_id, clinician_diagnosis, ...]
```

In [ ]:
# --- Opzione A: carica da Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r "/content/drive/MyDrive/IL_TUO_DATASET" /content/ML-Bitto/data/

# --- Opzione B: carica uno zip direttamente ---
# from google.colab import files
# uploaded = files.upload()   # seleziona il tuo .zip
# import zipfile
# with zipfile.ZipFile(list(uploaded.keys())[0], 'r') as z:
#     z.extractall('data/')

print("Decommentare e modificare una delle opzioni sopra per caricare i dati reali.")

In [ ]:
# --- Lancio pipeline con dataset pubblico ---
# Modifica --public con il path corretto dopo aver caricato il dataset

# !python main.py \
#     --public data/public_dataset_1 \
#     --epochs 50 \
#     --lr 1e-4 \
#     --batch-size 32 \
#     --monitor recall \
#     --num-workers 2

print("Decommentare il comando sopra dopo aver caricato il dataset.")

In [ ]:
# --- Lancio pipeline con dataset clinico (modalità completa) ---

# !python main.py \
#     --public  data/public_dataset_1 \
#     --clinical data/clinical_dataset \
#     --epochs 50 \
#     --lr 1e-4 \
#     --batch-size 16 \
#     --monitor recall \
#     --num-workers 2

print("Decommentare il comando sopra dopo aver caricato entrambi i dataset.")

In [ ]:
# --- Lancio con Grad-CAM e Cross-Validation ---

# !python main.py \
#     --clinical data/clinical_dataset \
#     --epochs 50 \
#     --monitor recall \
#     --gradcam \
#     --cross-val \
#     --num-workers 2

print("Decommentare il comando sopra per abilitare Grad-CAM e cross-validation.")

---

## 7 · Salva risultati su Drive

I risultati in `evaluation_results/` e i checkpoint in `checkpoints/` vengono persi al termine della sessione Colab.  
Usa questa cella per copiarli su Google Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# import shutil, datetime
# timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
# dest = f"/content/drive/MyDrive/ML-Bitto-results-{timestamp}"

# shutil.copytree("evaluation_results", f"{dest}/evaluation_results", dirs_exist_ok=True)
# shutil.copytree("checkpoints", f"{dest}/checkpoints", dirs_exist_ok=True)
# print(f"✓ Risultati salvati in: {dest}")

print("Decommentare le righe sopra per salvare i risultati su Google Drive.")